# INGESTA DE DATOS

In [ ]:
import requests
from bs4 import BeautifulSoup


def get_nota_fecha(fecha): # RETORNA UNA LISTA DE ID'S PARA LAS NOTAS DE ESE DIA
  ids = []
  url = f"https://sidofqa.segob.gob.mx/dof/sidof/notas/{fecha}"
  respuesta = requests.get(url)
  if respuesta.status_code == 200:
    data = respuesta.json()
    #print(f"respuesta en formato json {data}")
    if len(data['NotasVespertinas']) != 0:
      print("vespertinas existentes")
    if len(data['NotasExtraordinarias']) != 0:
      print("Extraordinarias existentes")
    if len(data['NotasMatutinas']) != 0:
      print("Matutinas existentes")
      for nota in data['NotasMatutinas']:
        #print(nota['codNota'])
        ids.append(str(nota['codNota']))
      return ids
  else:
    print(f"Error al obtener los datos: {respuesta.status_code}")

def get_nota_cod(cods):
    DIC = {"notas": [], "error": []}
    for cod in cods:
        url = f"https://sidofqa.segob.gob.mx/dof/sidof/notas/nota/{cod}"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            
            contenido_html = data['Nota'].get('cadenaContenido')
            soup = BeautifulSoup(contenido_html, 'lxml') if contenido_html else None
            texto = soup.get_text(separator=' ', strip=True) if soup else None
            titulo = data['Nota'].get('titulo', None)
            
            if not texto and data['Nota'].get('cadenaContenido') != None:
                DIC['error'].append([data['Nota']['pagina'],cod])
            else:
                organismos = [data['Nota'][orga] for orga in ['codOrgaUno', 'codOrgaDos', 'codOrgaTres', 'codOrgaCuatro'] if  data['Nota'].get(orga) and data['Nota'][orga] != 'null']
                DIC['notas'].append({'titulo': titulo,'texto': texto,'organismos': organismos})
        else:
            print(f"Error al descargar el documento DOC {cod}: {response.status_code}")
            DIC['error'].append([cod,data['Nota']['codDiario']])

    return DIC

notas_cod = get_nota_fecha("03-04-2024")

dic = get_nota_cod(notas_cod)
print(dic['error'])

In [ ]:
count = 0
for n in dic['notas'][:]:
  print(f" TITULO = {n['titulo']}\n TEXTO = {n['texto']}\n org = {n['organismos']}")
  count = count + 1
print(f" notas conseguidas {count}")
for e in dic['error']:
  print(f"nota {e[0]} en diario {e[1]}")

## Obtencion de Diarios en formato pdf
 Con el uso de la api del sidof. Obtenemos Diarios asociados a un dia, un diario es la coleccion de varias notas.

In [ ]:


def get_diario_doc(id_diario): # Solucionar error de la respuesta del http, siempre manda a error 404
  for id in id_diario:
    url = f"https://sidofqa.segob.gob.mx/dof/sidof/documentos/doc/{id}"
    response = requests.get(url)
    if response.status_code == 200:
        with open(f"diario_{id}.doc", "wb") as file:
            file.write(response.content)
            print(f"Documento DOC descargado correctamente: diario_{id}.doc")
    else:
        print(f"Error al descargar el documento DOC {id}: {response.status_code}")



def get_diario_pdf(id_diario): # FALTA PARSEAR SI ES UNA LISTA DE IDS
  for id in id_diario:
    url = f"https://sidofqa.segob.gob.mx/dof/sidof/documentos/pdf/{id}"
    response = requests.get(url)
    if response.status_code == 200:
        with open(f"diario_{id}.pdf", "wb") as file:
            file.write(response.content)
        print("Documento PDF descargado correctamente.")
    else:
        print(f"Error al descargar el documento PDF: {response.status_code}")


def get_diario_fecha(fecha): # RETORNA UNA LISTA DE ID'S PARA LOS DOCUMENTOS DE ESE DIA
  id = []
  url = f"https://sidofqa.segob.gob.mx/dof/sidof/diarios/porFecha/{fecha}"
  respuesta = requests.get(url)
  if respuesta.status_code == 200:
    data = respuesta.json()
    print(f"respuesta en formato json {data}")
    if data['Vespertina'] is not None:
      id.append(data['Vespertina'][0]['codDiario'])
    if data['Extraordinaria'] is not None:
      id.append(data['Extraordinaria'][0]['codDiario'])
    if data['Matutina'] is not None:
      id.append(data['Matutina'][0]['codDiario'])
    print(id)
    return id
  else:
    print(f"Error al obtener los datos: {respuesta.status_code}")


def get_diario_year(year): # RETORNA UNA LISTA DE DOCUMENTOS DE ESE YEAR
  ids = []
  url = f"https://sidofqa.segob.gob.mx/dof/sidof/diarios/{year}"
  respuesta = requests.get(url)
  if respuesta.status_code == 200:
    data = respuesta.json()
    print(f"respuesta en formato json {data}")
    for dic in data['ListaDiarios']:
      #print(dic['codDiario'])
      ids.append(dic['codDiario'])
    return ids
  else:
    print(f"Error al obtener los datos: {respuesta.status_code}")



get_diario_pdf(["312501"])
